# ***Events Normalization(Loopless)***

This notebook contains the code used to **normalize GA4 events data** by flattening the `event_params` and `user_properties` columns without using loops.

In [1]:
import json
import pandas as pd
import time
import datetime
import orjson

In [2]:
with open('../../raw_data/events.json', 'r') as f:
    data = json.load(f)

In [3]:
data_df = pd.json_normalize(data,sep='_')

In [4]:
data_df.head()

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,...,session_traffic_source_last_click_dv360_campaign,collected_traffic_source,session_traffic_source_last_click_google_ads_campaign_customer_id,session_traffic_source_last_click_google_ads_campaign_account_name,session_traffic_source_last_click_google_ads_campaign_campaign_id,session_traffic_source_last_click_google_ads_campaign_campaign_name,session_traffic_source_last_click_google_ads_campaign_ad_group_id,session_traffic_source_last_click_google_ads_campaign_ad_group_name,user_ltv_revenue,user_ltv_currency
0,20241113,1731513971041603,first_visit,"[{'key': 'batch_page_id', 'value': {'string_va...",None,None,-2094340797,None,None,2091574202.1731513971,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20241113,1731513971041603,session_start,"[{'key': 'ga_session_id', 'value': {'string_va...",None,None,-2094340797,None,None,2091574202.1731513971,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20241113,1731513971041603,page_view,"[{'key': 'ga_session_number', 'value': {'strin...",None,None,-2094340797,None,None,2091574202.1731513971,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20241113,1731513976070304,user_session_info,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,-2089312096,None,None,2091574202.1731513971,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20241113,1731513976070304,scroll,"[{'key': 'page_location', 'value': {'string_va...",None,None,-2089312096,None,None,2091574202.1731513971,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def normalize_params(df, column_name, prefix):
    df_column = df[[column_name]].copy()
    exploded = df_column.explode(column_name).reset_index(names='original_index')
    params_df = pd.json_normalize(exploded[column_name])
    
    exploded['key']=params_df['key']
    exploded['value']=(
        params_df['value.string_value']
        .fillna(params_df['value.int_value'])
        .fillna(params_df['value.float_value'])
        .fillna(params_df['value.double_value'])
    )
    
    normalized_df = (
        exploded
        .pivot(index='original_index',columns='key',values='value')
        .reset_index()
        .sort_index(axis=1)
    )
    normalized_df = normalized_df.add_prefix(prefix)
    normalized_df = normalized_df.rename(columns={f"{prefix}original_index": "original_index"})
    
    return normalized_df

In [6]:
def merge_normalized_dfs(base_df, *normalized_dfs):
    base = base_df.reset_index(drop=True)

    dfs = [base]
    for df in normalized_dfs:
        if 'original_index' in df.columns:
            df = df.sort_values('original_index').drop(columns=['original_index'], errors='ignore').reset_index(drop=True)
        dfs.append(df)

    merged = pd.concat(dfs, axis=1)
    merged.drop(columns=['event_params', 'user_properties'], errors='ignore', inplace=True)

    return merged

In [7]:
event_parameters = normalize_params(data_df, 'event_params', 'ep_')

In [8]:
event_parameters

key,ep_batch_ordering_id,ep_batch_page_id,ep_campaign,ep_click_classes,ep_click_id,ep_click_text,ep_click_url,ep_content,ep_engaged_session_event,ep_engagement_time_msec,...,ep_medium,original_index,ep_page_location,ep_page_referrer,ep_page_title,ep_percent_scrolled,ep_session_engaged,ep_source,ep_term,ep_value
0,1.0,1731513969906.0,productpickup,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,4pc,0,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,prm,NaN,NaN
1,1.0,1731513969906.0,productpickup,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,4pc,1,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,1.0,prm,NaN,NaN
2,1.0,1731513969906.0,productpickup,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,4pc,2,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,prm,NaN,NaN
3,2.0,1731513969906.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,158.0,...,NaN,3,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,NaN,NaN,NaN
4,2.0,1731513969906.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1120.0,...,NaN,4,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,10.0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,31.0,...,NaN,260859,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,NaN,0,NaN,NaN,NaN
260860,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3255.0,...,NaN,260860,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,10.0,0,NaN,NaN,NaN
260861,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,...,NaN,260861,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,20.0,0,NaN,NaN,NaN
260862,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,NaN,260862,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,30.0,0,NaN,NaN,NaN


In [9]:
user_properties = normalize_params(data_df, 'user_properties', 'user_prop_')

In [10]:
user_properties

key,original_index,user_prop_user_client_id,user_prop_user_session_id,user_prop_nan
0,0,NaN,NaN,NaN
1,1,NaN,NaN,NaN
2,2,NaN,NaN,NaN
3,3,_2091574202.1731513971,_1731513970,NaN
4,4,_2091574202.1731513971,_1731513970,NaN
...,...,...,...,...
260859,260859,_1436230623.1728949201,_1731552186,NaN
260860,260860,_1436230623.1728949201,_1731552186,NaN
260861,260861,_1436230623.1728949201,_1731552186,NaN
260862,260862,_1436230623.1728949201,_1731552186,NaN


In [11]:
final_df = merge_normalized_dfs(data_df, event_parameters, user_properties)

In [12]:
final_df

,event_date,event_timestamp,event_name,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,user_first_touch_timestamp,...,ep_page_referrer,ep_page_title,ep_percent_scrolled,ep_session_engaged,ep_source,ep_term,ep_value,user_prop_user_client_id,user_prop_user_session_id,user_prop_nan
0,20241113,1731513971041603,first_visit,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,prm,NaN,NaN,NaN,NaN,NaN
1,20241113,1731513971041603,session_start,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,1.0,prm,NaN,NaN,NaN,NaN,NaN
2,20241113,1731513971041603,page_view,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,prm,NaN,NaN,NaN,NaN,NaN
3,20241113,1731513976070304,user_session_info,None,None,-2089312096,None,None,2091574202.1731513971,1.731514e+15,...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,NaN,NaN,NaN,_2091574202.1731513971,_1731513970,NaN
4,20241113,1731513976070304,scroll,None,None,-2089312096,None,None,2091574202.1731513971,1.731514e+15,...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,10.0,0,NaN,NaN,NaN,_2091574202.1731513971,_1731513970,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,Eye Exam Center Locations Near You,NaN,0,NaN,NaN,NaN,_1436230623.1728949201,_1731552186,NaN
260860,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,Eye Exam Center Locations Near You,10.0,0,NaN,NaN,NaN,_1436230623.1728949201,_1731552186,NaN
260861,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,Eye Exam Center Locations Near You,20.0,0,NaN,NaN,NaN,_1436230623.1728949201,_1731552186,NaN
260862,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,Eye Exam Center Locations Near You,30.0,0,NaN,NaN,NaN,_1436230623.1728949201,_1731552186,NaN
